In [ ]:
#!/usr/bin/env python3
"""
CSV数据分析工具 - 使用大模型生成分析代码并输出HTML报告
"""

import os
import re
import subprocess
import tempfile
from pathlib import Path
from openai import OpenAI
import pandas as pd
import sys

# 获取当前目录
current_dir = os.getcwd()

class CSVAnalyzer:

    """基于大模型的CSV数据分析器"""
    
    def __init__(self, model_name: str = "gpt-4o"):
        """
        初始化分析器
        
        参数:
            model_name: 使用的模型名称
        """
        self.client = OpenAI()
        self.model_name = model_name
    
    def _call_llm(self, prompt: str) -> str:
        """
        调用大模型
        
        参数:
            prompt: 提示词
            
        返回:
            模型响应内容
        """
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": "你是一个专业的数据分析和代码生成助手。"},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=4000
        )
        return response.choices[0].message.content
    
    def _extract_code(self, llm_response: str, language: str = "python") -> str:
        """
        从LLM响应中提取代码块
        
        参数:
            llm_response: LLM的响应内容
            language: 代码语言类型
            
        返回:
            提取的代码
        """
        # 尝试匹配代码块
        pattern = rf"```{language}\s*\n(.*?)```"
        matches = re.findall(pattern, llm_response, re.DOTALL)
        
        if matches:
            return matches[0].strip()
        
        # 如果没有找到代码块标记，尝试直接返回内容
        # 去除可能的markdown标记
        code = llm_response.strip()
        if code.startswith("```"):
            lines = code.split("\n")
            code = "\n".join(lines[1:-1]) if len(lines) > 2 else code
        
        return code.strip()
    
    def generate_analysis_code(self, csv_path: str, user_intent: str) -> str:
        """
        生成数据分析代码
        
        参数:
            csv_path: CSV文件路径
            user_intent: 用户分析意图
            
        返回:
            生成的Python代码
        """
        prompt = f"""你是一个专业的数据分析助手。用户提供了一个CSV文件，需要你生成Python代码来分析数据。

CSV文件路径：{csv_path}
用户分析意图：{user_intent}

请生成完整的Python代码，要求：

【数据读取】
1. 使用pandas读取CSV文件（确保处理编码问题，尝试utf-8、gbk、gb2312）
2. 读取后立即打印列名，确认数据结构
3. 不要假设列名，使用实际读取到的列名进行分析

【数据探索】
4. 展示数据基本信息：
   - 数据形状（行数、列数）
   - 实际的列名列表
   - 如果有缺失值，统计缺失值的数量，没有则不用体现
   - 前5行数据预览
5. 识别时间列（可能是"日期"、"时间"、"date"、"time"等），并转换为datetime类型
6. 识别数值列和分类列

【数据分析】
7. **重要**：基于实际存在的列进行分析，不要编造不存在的列
8. 根据用户意图进行针对性分析：
   - 如果有时间列：分析时间趋势、增长率、周期性
   - 如果有数值列：计算统计量（总和、均值、中位数、最大最小值）
   - 如果有分类列：分组统计、排名、占比分析
   - 如果有多个维度：交叉分析、对比分析
9. 提取3-5个关键发现和洞察

【报告生成】
10. 将分析结果组织成结构化的文字报告，存储在变量 `analysis_report` 中
11. **重要**：构建报告时使用字符串拼接，不要使用f-string格式化pandas对象
    - 正确做法：先计算值，再格式化
    - 报告使用的语言一致，只支持中文或英文，不支持中英文混合
    - 错误做法：直接在f-string中格式化DataFrame或Series
    - 使用 .values.tolist() 或 .to_dict() 转换pandas对象为Python原生类型
12. 报告结构（使用清晰的分隔符）：
    ```
    [主标题]

    ===== 数据概览 =====
    [列名等基本信息，根据实际情况出发]
    
    ===== 关键发现 =====
    [3-5个核心洞察，每个洞察单独一段]
    
    ===== 详细分析 =====
    [针对用户意图的深入分析，包含具体数据]
    
    ===== 结论与建议 =====
    [总结和可操作的建议]
    
    ```
13. 代码最后必须打印 `analysis_report` 的内容

【代码质量】
14. 使用try-except处理异常（特别是编码问题和数据类型转换）
15. 添加必要的注释说明关键步骤
16. 不要生成图表文件或保存任何文件
17. 确保代码可以独立运行
18. 使用 str() 或 .item() 方法将numpy/pandas标量转换为Python原生类型

【代码示例参考】
正确的数据格式化方式：
- 先计算数值：total = df['销售额'].sum()
- 再格式化输出：report += f"总销售额: {{total:,.0f}}元\\n"
- 分组统计使用.items()遍历
- 日期转换后使用.strftime()格式化

请只返回可执行的Python代码，不要包含任何解释文字。代码应该以```python开始，以```结束。"""

        llm_response = self._call_llm(prompt)
        code = self._extract_code(llm_response, "python")
        return code
    
    def execute_analysis_code(self, code: str) -> str:
        """
        执行分析代码并获取结果
        
        参数:
            code: Python代码
            
        返回:
            执行结果（分析报告）
        """
        # 创建临时文件保存代码
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False, encoding='utf-8') as f:
            f.write(code)
            temp_file = f.name
        
        try:
            # 执行代码并捕获输出
            result = subprocess.run(
                [sys.executable, temp_file],
                capture_output=True,
                text=True,
                timeout=60,
                encoding='utf-8'
            )
            
            if result.returncode != 0:
                raise RuntimeError(f"代码执行失败:\n{result.stderr}")
            
            return result.stdout.strip()
        
        finally:
            # 清理临时文件
            if os.path.exists(temp_file):
                os.remove(temp_file)
    
    def generate_html_code(self, analysis_report: str, user_intent: str) -> str:
        """
        生成HTML报告代码
        
        参数:
            analysis_report: 分析报告内容
            user_intent: 用户原始意图
            
        返回:
            生成的HTML代码
        """
        prompt = f"""你是一个专业的前端开发工程师。现在需要你生成一个HTML报告页面。

数据分析结论：
{analysis_report}

用户原始意图：{user_intent}

请生成完整的HTML代码，要求：
1. 使用现代化的CSS样式，页面美观专业
2. 必须包含以下部分：
   - 页面明确的标题和副标题，字体清晰，设计美观大方
   - 分析概览卡片
   - 详细分析内容（使用合适的排版）
   - 关键发现加粗显示
   - 结论和建议部分，字数大于100字
3. 使用响应式设计，适配不同屏幕
4. 使用内联CSS或者<style>标签，不要依赖外部文件
5. 配色专业，建议使用蓝色或深色主题
6. 结构清晰，易于阅读
7. 可以使用适当的图标或装饰元素（使用Unicode符号或CSS实现）
8. 不要在页面显示任何代码块
9. 所有数据图表都不能超出页面大小，超出则需要自适应缩小

请只返回完整的HTML代码，不要包含任何解释文字。代码应该以```html开始，以```结束。"""

        llm_response = self._call_llm(prompt)
        html_code = self._extract_code(llm_response, "html")
        return html_code
    
    def analyze_csv_to_html(
        self, 
        csv_path: str, 
        user_intent: str, 
        output_html_path: str = None
    ) -> str:
        """
        分析CSV文件并生成HTML报告（主函数）
        
        参数:
            csv_path: CSV文件的本地路径
            user_intent: 用户的分析意图描述
            output_html_path: 输出HTML文件路径（可选，默认自动生成）
        
        返回:
            生成的HTML文件路径
        """
        print("=" * 60)
        print("CSV数据分析工具 - 启动")
        print("=" * 60)
        
        # 验证CSV文件存在
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"CSV文件不存在: {csv_path}")
        
        print(f"\n[1/4] 输入信息")
        print(f"  - CSV文件: {csv_path}")
        print(f"  - 分析意图: {user_intent}")
        
        # 第一步：生成分析代码
        print(f"\n[2/4] 生成数据分析代码...")
        analysis_code = self.generate_analysis_code(csv_path, user_intent)
        print(f"  ✓ 代码生成完成 ({len(analysis_code)} 字符)")
        
        # 执行分析代码
        print(f"\n[3/4] 执行分析代码...")
        analysis_report = self.execute_analysis_code(analysis_code)
        print(f"  ✓ 分析完成 ({len(analysis_report)} 字符)")
        print(f"\n--- 分析报告预览 ---")
        print(analysis_report[:500] + "..." if len(analysis_report) > 500 else analysis_report)
        print(f"--- 预览结束 ---\n")
        
        # 第二步：生成HTML代码
        print(f"[4/4] 生成HTML报告...")
        html_code = self.generate_html_code(analysis_report, user_intent)
        print(f"  ✓ HTML代码生成完成 ({len(html_code)} 字符)")
        
        # 保存HTML文件
        if output_html_path is None:
            csv_basename = Path(csv_path).stem
            current_dir = Path.cwd()  # 当前工作目录
            output_html_path = current_dir / f"{csv_basename}_report.html"
        
        with open(output_html_path, 'w', encoding='utf-8') as f:
            f.write(html_code)
        
        print(f"\n✓ HTML报告已保存: {output_html_path}")
        print("=" * 60)
        
        return output_html_path


def main():
    """命令行入口示例"""
    import sys
    
    if len(sys.argv) < 3:
        print("用法: python csv_analyzer.py <CSV文件路径> <分析意图> [输出HTML路径]")
        print("\n示例:")
        print('  python csv_analyzer.py data.csv "分析数据趋势"')
        print('  python csv_analyzer.py data.csv "找出异常数据" output.html')
        sys.exit(1)
    
    csv_path = sys.argv[1]
    user_intent = sys.argv[2]
    output_path = sys.argv[3] if len(sys.argv) > 3 else None
    
    analyzer = CSVAnalyzer()
    result_path = analyzer.analyze_csv_to_html(csv_path, user_intent, output_path)
    
    print(f"\n完成！HTML报告路径: {result_path}")



#if __name__ == "__main__":
#    main()
    
if __name__ == "__main__":
 
 
 ##需填写模型的的key和url！！！！！！！！！！！！！!!!
    os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"
    os.environ["OPENAI_BASE_URL"] = "YOUR_API_URL"



    analyzer = CSVAnalyzer()
    csv_file = os.path.join(current_dir, "demo_data.csv")
    result_path = analyzer.analyze_csv_to_html(csv_file, "分析数据生成报告", None)




CSV数据分析工具 - 启动

[1/4] 输入信息
  - CSV文件: /Users/anglekaka/Desktop/study_ai/data_analysis_demo_20251022/demo_data.csv
  - 分析意图: 分析数据生成报告

[2/4] 生成数据分析代码...
  ✓ 代码生成完成 (4049 字符)

[3/4] 执行分析代码...
  ✓ 分析完成 (3250 字符)

--- 分析报告预览 ---
Successfully read the file with encoding: utf-8
Column names: ['股票名称', '股票代码', '日期', '开盘', '收盘', '最高', '最低', '成交量', '成交额', '振幅', '涨跌幅', '涨跌额', '换手率']
Data Shape: (240, 13)
Actual Column Names: ['股票名称', '股票代码', '日期', '开盘', '收盘', '最高', '最低', '成交量', '成交额', '振幅', '涨跌幅', '涨跌额', '换手率']
Data Preview:
    股票名称    股票代码                日期       开盘  ...    振幅   涨跌幅   涨跌额   换手率
0  贵州茅台  600519  2025-10-22 09:31  1462.08  ...  0.31  0.05  0.74  0.01
1  贵州茅台  600519  2025-10-22 09:32  1463.00  ...  0.27 -0.02 ...
--- 预览结束 ---

[4/4] 生成HTML报告...
  ✓ HTML代码生成完成 (4683 字符)

✓ HTML报告已保存: /Users/anglekaka/Desktop/study_ai/data_analysis_demo_20251022/demo_data_report.html
